In [11]:
# =========================
# 📦 Step 1: Imports
# =========================
import pandas as pd
from pathlib import Path


# =========================
# 📂 Step 2: Load Data
# =========================
BASE_DIR = Path().resolve().parent
DATA_PATH = BASE_DIR / "data" / "processed" / "neighbourhood_nhanes_demographics_2003_2014_appended.csv"

df = pd.read_csv(DATA_PATH)

print("Raw Data Preview:")
display(df.head(10))


# =========================
# 🧹 Step 3: Standardize Column Names
# =========================
df.columns = df.columns.str.upper()


# =========================
# 🧠 Step 3: Create Recoding Function
# =========================
def recode_with_missing(series, mapping):
    """
    Keeps original data intact.
    Converts '.', 7, 9 → Missing in recoded column only.
    """
    cleaned = series.replace({
        ".": pd.NA
    })
    return cleaned.map(mapping).fillna("Missing")


# =========================
# ⚠️ Step 4: Create Clean Copy for Recoding
# =========================
df_recoded = df.copy()


# =========================
# 🧠 Step 5: Add Recoded Columns
# =========================

# -------- Gender --------
df_recoded["RIAGENDR_RECODE"] = df_recoded["RIAGENDR"].map({
    1: "Male",
    2: "Female"
})


# -------- Race --------
df_recoded["RIDRETH1_RECODE"] = df_recoded["RIDRETH1"].map({
    1: "Mexican American",
    2: "Other Hispanic",
    3: "Non-Hispanic White",
    4: "Non-Hispanic Black",
    5: "Other Race"
})


# -------- Education --------
df_recoded["DMDEDUC2_RECODE"] = df_recoded["DMDEDUC2"].map({
    1: "< 9th grade",
    2: "9-11th grade",
    3: "High school / GED",
    4: "Some college / AA",
    5: "College graduate or above",
    7: "Refused",
    9: "Don't Know"
})


# -------- Marital Status (optional but useful) --------
if "DMDMARTL" in df_recoded.columns:
    df_recoded["DMDMARTL_RECODE"] = df_recoded["DMDMARTL"].map({
        1: "Married",
        2: "Widowed",
        3: "Divorced",
        4: "Separated",
        5: "Never married",
        6: "Living with partner",
        77: "Refused",
        99: "Don't know"
    })


# =========================
# 👀 Step 6: Preview (Original + Recoded Together)
# =========================
cols_preview = [
    "SEQN",
    "RIAGENDR", "RIAGENDR_RECODE",
    "RIDRETH1", "RIDRETH1_RECODE",
    "DMDEDUC2", "DMDEDUC2_RECODE",
    "DMDMARTL","DMDMARTL_RECODE"
]

cols_preview = [c for c in cols_preview if c in df_recoded.columns]
display(df_recoded[cols_preview].head(10))


# =========================
# 📊 Step 7: Validation
# =========================
print("Final shape:", df_recoded.shape)
print("\nMissing values in recoded columns:")
for col in df_recoded.columns:
    if col.endswith("_RECODE"):
        print(col, ":", df_recoded[col].isnull().sum())



# =========================
# 🔁 Step 8: FINAL RENAME (DO THIS LAST)
# =========================
# 44 ORIGINAL VARIABLES AND 4 RECODED VARIABLES

rename_map = {
    # =========================
    # IDs / Survey structure
    # =========================
    "SEQN": "SEQN_participant_id",
    "SDDSRVYR": "SDDSRVYR_survey_cycle",
    "RIDSTATR": "RIDSTATR_interview_exam_status_code",
    "RIDEXMON": "RIDEXMON_exam_period_code",
    # =========================
    # Demographics
    # =========================
    "RIAGENDR": "RIAGENDR_gender",
    "RIAGENDR_RECODE" : "RIAGENDR_gender_recode",
    "RIDAGEYR": "RIDAGEYR_age_years",
    "RIDAGEMN": "RIDAGEMN_age_months",
    "RIDAGEEX": "RIDAGEEX_age_exam_months",
    "RIDRETH1": "RIDRETH1_race_ethnicity",
    "RIDRETH1_RECODE" :"RIDRETH1_race_ethnicity_recode",
    "RIDRETH2": "RIDRETH2_race_ethnicity_linked",
    "DMDBORN": "DMDBORN_country_of_birth",
    "DMDCITZN": "DMDCITZN_citizenship_status",
    "DMDYRSUS": "DMDYRSUS_years_in_us",
    
    # =========================
    # Education
    # =========================
    "DMDEDUC3": "DMDEDUC3_education_children",
    "DMDEDUC2": "DMDEDUC2_education_adult",
    "DMDEDUC2_RECODE":"DMDEDUC2_education_adult_recode",
    "DMDEDUC": "DMDEDUC_education",
    "DMDSCHOL": "DMDSCHOL_school_attendance",

    # =========================
    # Marital / Household
    # =========================
    "DMDMARTL": "DMDMARTL_marital_status",
    "DMDMARTL_RECODE": "DMDMARTZ_marital_status_recode",
    "DMDHHSIZ": "DMDHHSIZ_household_size",
    "INDHHINC": "INDHHINC_household_income",
    "INDFMINC": "INDFMINC_family_income",
    "INDFMPIR": "INDFMPIR_income_poverty_ratio",

    # =========================
    # Pregnancy / Health status
    # =========================
    "RIDEXPRG": "RIDEXPRG_pregnancy_status_code",
    "DMQMILIT": "DMQMILIT_military_veteran_status_code",

    # =========================
    # Household reference person
    # =========================
    "DMDHRGND": "DMDHRGND_hh_ref_gender_code",
    "DMDHRAGE": "DMDHRAGE_hh_ref_age",
    "DMDHRBRN": "DMDHRBRN_hh_ref_birth_country_code",
    "DMDHREDU": "DMDHREDU_hh_ref_education_code",
    "DMDHRMAR": "DMDHRMAR_hh_ref_marital_status_code",
    "DMDHSEDU": "DMDHSEDU_hh_spouse_education_code",

    # =========================
    # Interview metadata
    # =========================
    "SIALANG": "SIALANG_interview_language_code",
    "SIAPROXY": "SIAPROXY_interview_proxy_code",
    "SIAINTRP": "SIAINTRP_interview_interpreter_code",
    "FIALANG": "FIALANG_family_interview_language_code",
    "FIAPROXY": "FIAPROXY_family_proxy_code",
    "FIAINTRP": "FIAINTRP_family_interpreter_code",
    "MIALANG": "MIALANG_mec_interview_language_code",
    "MIAPROXY": "MIAPROXY_mec_proxy_code",
    "MIAINTRP": "MIAINTRP_mec_interpreter_code",
    "AIALANG": "AIALANG_acasi_language_code",

    # =========================
    # Weights / Survey design
    # =========================
    "WTINT2YR": "WTINT2YR_interview_weight_2yr",
    "WTMEC2YR": "WTMEC2YR_mec_exam_weight_2yr",
    "SDMVPSU": "SDMVPSU_psu",
    "SDMVSTRA": "SDMVSTRA_strata"
}

df_renamed = df_recoded.rename(columns=rename_map)

# =========================
# 👀 Step 9: Final Check
# =========================
print("Final shape:", df_renamed.shape)
display(df_renamed.head())


# =========================
# 💾 Step 8: Save Output
# =========================
OUTPUT_DIR = BASE_DIR / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df_recoded.to_csv(OUTPUT_DIR / "neighbourhood_nhanes_demographics_2003_2014_appended_cleaned.csv",index=False)
df_renamed.to_csv(OUTPUT_DIR / "neighbourhood_nhanes_demographics_2003_2014_appended_cleaned_renamed.csv",index=False)


print("Saved files:")
print(OUTPUT_DIR / "neighbourhood_nhanes_demographics_2003_2014_appended_cleaned.csv")
print(OUTPUT_DIR / "neighbourhood_nhanes_demographics_2003_2014_appended_cleaned_renamed.csv")


Raw Data Preview:


,SEQN,SDDSRVYR,RIDSTATR,RIDEXMON,RIAGENDR,RIDAGEYR,RIDAGEMN,RIDAGEEX,RIDRETH1,RIDRETH2,...,RIDEXAGY,RIDEXAGM,DMQMILIZ,DMQADFC,DMDBORN4,AIALANGA,DMDHHSZA,DMDHHSZB,DMDHHSZE,DMDHRBR4
0,21005.0,3.0,2.0,1.0,1.0,1.900000e+01,232.0,233.0,4.0,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,21006.0,3.0,2.0,2.0,2.0,1.600000e+01,203.0,205.0,4.0,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,21007.0,3.0,2.0,1.0,2.0,1.400000e+01,172.0,172.0,3.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,21008.0,3.0,2.0,2.0,1.0,1.700000e+01,208.0,209.0,4.0,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,21009.0,3.0,2.0,2.0,1.0,5.500000e+01,671.0,672.0,3.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,21010.0,3.0,2.0,2.0,2.0,5.200000e+01,633.0,634.0,3.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,21011.0,3.0,2.0,1.0,1.0,5.397605e-79,3.0,4.0,1.0,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,21012.0,3.0,2.0,2.0,1.0,6.300000e+01,765.0,766.0,4.0,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,21013.0,3.0,2.0,1.0,2.0,1.300000e+01,163.0,164.0,4.0,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,21014.0,3.0,2.0,2.0,1.0,3.000000e+00,42.0,42.0,4.0,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,SEQN,RIAGENDR,RIAGENDR_RECODE,RIDRETH1,RIDRETH1_RECODE,DMDEDUC2,DMDEDUC2_RECODE,DMDMARTL,DMDMARTL_RECODE
0,21005.0,1.0,Male,4.0,Non-Hispanic Black,NaN,NaN,5.0,Never married
1,21006.0,2.0,Female,4.0,Non-Hispanic Black,NaN,NaN,5.0,Never married
2,21007.0,2.0,Female,3.0,Non-Hispanic White,NaN,NaN,5.0,Never married
3,21008.0,1.0,Male,4.0,Non-Hispanic Black,NaN,NaN,5.0,Never married
4,21009.0,1.0,Male,3.0,Non-Hispanic White,3.0,High school / GED,1.0,Married
5,21010.0,2.0,Female,3.0,Non-Hispanic White,4.0,Some college / AA,6.0,Living with partner
6,21011.0,1.0,Male,1.0,Mexican American,NaN,NaN,NaN,NaN
7,21012.0,1.0,Male,4.0,Non-Hispanic Black,3.0,High school / GED,6.0,Living with partner
8,21013.0,2.0,Female,4.0,Non-Hispanic Black,NaN,NaN,NaN,NaN
9,21014.0,1.0,Male,4.0,Non-Hispanic Black,NaN,NaN,NaN,NaN


Final shape: (61087, 65)

Missing values in recoded columns:
RIAGENDR_RECODE : 0
RIDRETH1_RECODE : 0
DMDEDUC2_RECODE : 27585
DMDMARTL_RECODE : 24141
Final shape: (61087, 65)


,SEQN_participant_id,SDDSRVYR_survey_cycle,RIDSTATR_interview_exam_status_code,RIDEXMON_exam_period_code,RIAGENDR_gender,RIDAGEYR_age_years,RIDAGEMN_age_months,RIDAGEEX_age_exam_months,RIDRETH1_race_ethnicity,RIDRETH2_race_ethnicity_linked,...,DMDBORN4,AIALANGA,DMDHHSZA,DMDHHSZB,DMDHHSZE,DMDHRBR4,RIAGENDR_gender_recode,RIDRETH1_race_ethnicity_recode,DMDEDUC2_education_adult_recode,DMDMARTZ_marital_status_recode
0,21005.0,3.0,2.0,1.0,1.0,19.0,232.0,233.0,4.0,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,Male,Non-Hispanic Black,NaN,Never married
1,21006.0,3.0,2.0,2.0,2.0,16.0,203.0,205.0,4.0,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,Female,Non-Hispanic Black,NaN,Never married
2,21007.0,3.0,2.0,1.0,2.0,14.0,172.0,172.0,3.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,Female,Non-Hispanic White,NaN,Never married
3,21008.0,3.0,2.0,2.0,1.0,17.0,208.0,209.0,4.0,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,Male,Non-Hispanic Black,NaN,Never married
4,21009.0,3.0,2.0,2.0,1.0,55.0,671.0,672.0,3.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,Male,Non-Hispanic White,High school / GED,Married


Saved files:
C:\Users\NVK00\Desktop\MyData\Projects\Clinical_Neighbourhood_ML_Data_Predictor\data\processed\neighbourhood_nhanes_demographics_2003_2014_appended_cleaned.csv
C:\Users\NVK00\Desktop\MyData\Projects\Clinical_Neighbourhood_ML_Data_Predictor\data\processed\neighbourhood_nhanes_demographics_2003_2014_appended_cleaned_renamed.csv
